# Entrega 2 proyecto aprendizaje de máquina

Pedro Pablo Sanín Trujillo (202221527)


## 1 - Carga de datos


In [62]:
# Importar la librerías del proyecto
import pandas as pd
import tensorflow as tf
from keras.layers import TextVectorization
from sklearn.model_selection import train_test_split

In [63]:
# A continuación, se cargan los conjuntos de datos de entrenamiento y evaluación

path_train = "./datos/train.csv"
path_eval = "./datos/eval.csv"


dataset_train_original = pd.read_csv(path_train)
dataset_eval_original = pd.read_csv(path_eval)

dataset_train = dataset_train_original.copy()
dataset_eval = dataset_eval_original.copy()

In [64]:
print(
    f"Número de filas y columnas del conjunto de entrenamiento: {dataset_train.shape}"
)

dataset_train.head(10)

Número de filas y columnas del conjunto de entrenamiento: (20466, 3)


,id,text,label
0,0,Lo pedí un martes por internet y llegó al terc...,negativo
1,1,Les cuento como me fue. Compre esta tablet a m...,positivo
2,2,Lo compré hace como dos meses en una tienda de...,neutral
3,3,Le routeur capte très bien dans toute la maiso...,negativo
4,4,"I got it a while back on sale online, it arriv...",neutral
5,5,Comprei esse teclado mês passado pra trabalhar...,positivo
6,6,Comprei há umas duas semanas pela internet e c...,negativo
7,7,"La compre a mediados de año por internet, tard...",positivo
8,8,Les dejo un comentario rápido. Lo pedí a media...,negativo
9,9,The color accuracy on this monitor is genuinel...,positivo


Vemos que el dataset contiene 20466 filas, cada una con un id, la reseña correspondiente y un label que indica que si la reseña es positiva o negativa. Revisando manualmente algunas entradas, confirmamos que los únicos idiomas presentes en las reseñas son español, inglés, portugués, francés y una mezcla de español e inglés. Esto es consistente con lo indicado en el enunciado del proyecto. Preliminarmente, no separaremos los modelos reseñas por idioma pues esto reduce sustancialmente el tamaño de cada conjunto de datos. Sin embargo, si vale la pena identificar el idioma para tener una división train-test balanceada.

Además nuestra métrica de entrenamiento será el accuracy, de acuerdo con la competencia. A continuación, intentaremos identificar el idioma de cada una de nuestras reseñas.


## 2 - Identificando el idioma de la reseña

Para esto, descargaremos las 50 palabras más comunes en cada uno de los idiomas. Por la ley de Zipf, si las listas de las 50 palabras más comunes son distintas, una de las palabras aparecerá en la lista y esto nos servirá para determinar el idioma de la reseña. Asimismo, las palabras más comunes en cada idioma son artículos y generalmente tienen el mismo significado.

Así, el criterio para determinar el idioma de una reseña será el siguiente:

1. El primer y segundo idioma candidato serán los primeros dos conjuntos de palabras que tengan más coincidencia con las palabras de la reseña.

2. Con las palabras que pertenecen al conjunto más común,


In [65]:
import sys
import unicodedata
from collections import Counter

# Tabla de traducción que reemplaza por un espacio todo carácter Unicode de puntuación
# (categorías 'P*'), lo que incluye ¿ ¡ « » “ ” … además de la puntuación ASCII.
puntua = {
    i: " "
    for i in range(sys.maxunicode + 1)
    if unicodedata.category(chr(i)).startswith("P")
}

# Palabras de la lista de inglés que también son muy comunes en español ("lo he usado"),
# por lo que no cuentan como evidencia de inglés dentro de una reseña en español.
AMBIGUAS_EN = {"he"}


def procesar_resena(text):
    """Pasa la reseña a minúsculas, elimina la puntuación y la separa en palabras."""
    text = text.lower()
    text = text.translate(puntua)
    return text.split()


def palabras_exclusivas(palabras_por_idioma):
    """Para cada idioma, las palabras comunes que no aparecen en la lista de ningún otro idioma."""
    return {
        idioma: palabras
        - set().union(
            *(otras for otro, otras in palabras_por_idioma.items() if otro != idioma)
        )
        for idioma, palabras in palabras_por_idioma.items()
    }


def identificar_idioma(resena, palabras_por_idioma, exclusivas):
    """Identifica el idioma de una reseña: 'es', 'en', 'fr', 'pt_br', 'es_en' (bilingüe) o None.

    1. El idioma principal es el que tiene más ocurrencias de sus palabras exclusivas. Si la reseña
       no tiene ninguna palabra exclusiva, se usan todas las palabras comunes (incluidas las compartidas).
    2. Si el idioma principal es español o inglés y la reseña contiene al menos una palabra exclusiva
       del otro idioma, la reseña se considera bilingüe ('es_en').
    """
    conteo = Counter(procesar_resena(resena))

    def puntajes(listas):
        return {
            idioma: sum(n for palabra, n in conteo.items() if palabra in palabras)
            for idioma, palabras in listas.items()
        }

    puntaje = puntajes(exclusivas)
    if max(puntaje.values()) == 0:
        puntaje = puntajes(palabras_por_idioma)
        if max(puntaje.values()) == 0:
            return None
    principal = max(puntaje, key=lambda k: puntaje[k])

    hay_es = any(palabra in exclusivas["es"] for palabra in conteo)
    hay_en = any(palabra in exclusivas["en"] - AMBIGUAS_EN for palabra in conteo)
    if principal in ("es", "en") and hay_es and hay_en:
        return "es_en"
    return principal

In [66]:
# Cargamos las 50 palabras más comunes por idioma y les aplicamos la misma limpieza que a las reseñas,
# para que entradas como "c'", "l'" o "'s" queden como "c", "l" y "s" y puedan coincidir con las reseñas.
path_palabras = "./datos/50_palabras_mas_comunes/50_palabras_mas_comunes.csv"

palabras_comunes = pd.read_csv(path_palabras)
palabras_comunes["palabra"] = palabras_comunes["palabra"].map(
    lambda p: procesar_resena(p)[0]
)

# Conjunto de palabras por idioma, y el subconjunto de palabras exclusivas de cada idioma
palabras_por_idioma = palabras_comunes.groupby("idioma")["palabra"].apply(set).to_dict()
exclusivas = palabras_exclusivas(palabras_por_idioma)

palabras_comunes.head()

,idioma,rank,palabra,frecuencia
0,en,1,you,28787591
1,en,2,i,27086011
2,en,3,the,22761659
3,en,4,to,17099834
4,en,5,a,14484562


In [67]:
print(
    identificar_idioma("Esta es una reseña de prueba", palabras_por_idioma, exclusivas)
)
print(identificar_idioma("This is a test review", palabras_por_idioma, exclusivas))
print(
    identificar_idioma("Ceci est une critique de test", palabras_por_idioma, exclusivas)
)
print(
    identificar_idioma("Esta é uma crítica de teste", palabras_por_idioma, exclusivas)
)
print(
    identificar_idioma(
        "Esta es una reseña de prueba en inglés: This is a test review",
        palabras_por_idioma,
        exclusivas,
    )
)
print(
    identificar_idioma(
        "Esta es una reseña de prueba en francés: Ceci est une critique de test",
        palabras_por_idioma,
        exclusivas,
    )
)

es
en
fr
pt_br
es_en
es


In [68]:
# Identificamos el idioma de cada reseña de entrenamiento y evaluación
dataset_train["idioma"] = dataset_train["text"].map(
    lambda t: identificar_idioma(t, palabras_por_idioma, exclusivas)
)
dataset_eval["idioma"] = dataset_eval["text"].map(
    lambda t: identificar_idioma(t, palabras_por_idioma, exclusivas)
)

dataset_train["idioma"].value_counts(dropna=False, normalize=True)

idioma
es       0.652497
en       0.114678
pt_br    0.114043
fr       0.102511
es_en    0.016271
Name: proportion, dtype: float64

In [69]:
dataset_train.head()

,id,text,label,idioma
0,0,Lo pedí un martes por internet y llegó al terc...,negativo,es
1,1,Les cuento como me fue. Compre esta tablet a m...,positivo,es
2,2,Lo compré hace como dos meses en una tienda de...,neutral,es
3,3,Le routeur capte très bien dans toute la maiso...,negativo,fr
4,4,"I got it a while back on sale online, it arriv...",neutral,en


Ahora, dividimos el conjunto de datos en prueba y entrenamiento estratificado para incluir una proporción representativa de cada idioma y de cada etiqueta. Para esto, usaremos `sklearn.train_test_split`.


In [70]:
# Codificamos las etiquetas de como enteros
etiquetas = sorted(
    dataset_train["label"].unique()
)  # ['negativo', 'neutral', 'positivo']
etiqueta_a_id = {e: i for i, e in enumerate(etiquetas)}


# División 90/10 estratificada por idioma y etiqueta a la vez: cada combinación
# (p. ej. "fr|neutral") conserva la misma proporción en entrenamiento y prueba.
estrato = dataset_train["idioma"].astype(str) + "|" + dataset_train["label"]

# train_test_split exige al menos 2 reseñas por estrato. Las combinaciones con una sola
# reseña (solo hay una reseña bilingüe es_en neutral) no se pueden dividir, así que van
# directamente a entrenamiento.
raros = estrato.map(estrato.value_counts()) < 2

train_df, test_df = train_test_split(
    dataset_train[~raros],
    test_size=0.1,
    random_state=42,
    stratify=estrato[~raros],
)
train_df = pd.concat([train_df, dataset_train[raros]])


def dataset_a_tf(df, shuffle):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["text"].values, df["label"].map(etiqueta_a_id).values)
    )
    if shuffle:
        ds = ds.shuffle(len(df), seed=42)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)


train_dataset = dataset_a_tf(train_df, shuffle=True)
test_dataset = dataset_a_tf(test_df, shuffle=False)

## 2 - Red neuronal

In [71]:
encoder_mlp = TextVectorization(output_mode="tf_idf")
encoder_mlp.adapt(train_df["text"])

In [72]:
model_mlp = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(), dtype=tf.string),
        encoder_mlp,
    
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(3),
    ]
)

model_mlp.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_6            │ (None, 19045)          │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 32)             │       609,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 609,571 (2.33 MB)

 Trainable params: 609,571 (2.33 MB)

 Non-trainable params: 0 (0.00 B)

In [73]:
model_mlp.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

Ahora para entrenar el modelo, usaremos ``tf.keras.callbacks.EarlyStopping``. Esto nos permite parar el entrenamiento cuando la pérdida de validación (`val_loss`) deja de disminuir, y restaurar los parámetros de la época con menor pérdida de validación. Usamos la pérdida y no el accuracy porque el accuracy de validación puede seguir subiendo levemente mientras el modelo ya se está sobreajustando (la pérdida sube porque el modelo se vuelve demasiado confiado en sus errores).

In [74]:
# En lugar de fijar el número de épocas, entrenamos hasta un máximo de 50 y detenemos el
# entrenamiento cuando la métrica `monitor` de validación deja de mejorar durante `patience`
# épocas seguidas. restore_best_weights=True deja el modelo con los pesos de la mejor época.
# Por defecto se monitorea la pérdida de validación ("val_loss"), que se minimiza; si se pasa
# un accuracy (p. ej. monitor="val_accuracy"), se maximiza.
def entrenar_modelo_con_parada_temprana(
    model, train_dataset, test_dataset, max_epochs=50, patience=3, monitor="val_loss"
):
    modo = "min" if "loss" in monitor else "max"
    parada_temprana = tf.keras.callbacks.EarlyStopping(
        monitor=monitor,
        mode=modo,
        patience=patience,
        restore_best_weights=True,
        verbose=1,  # informa en qué época se detuvo y de qué época restauró los pesos
    )

    history = model.fit(
        train_dataset,
        epochs=max_epochs,
        validation_data=test_dataset,
        callbacks=[parada_temprana],
    )

    # Guardamos en el historial la época cuyos pesos restauró la parada temprana (índice desde 0),
    # para que los resúmenes usen exactamente esa época y no la recalculen con otra métrica.
    history.mejor_epoca = parada_temprana.best_epoch
    print(
        f"Mejor época según {monitor}: {history.mejor_epoca + 1} "
        f"(val_loss = {history.history['val_loss'][history.mejor_epoca]:.4f}, "
        f"val_accuracy = {history.history['val_accuracy'][history.mejor_epoca]:.4f})"
    )

    return history

In [75]:
entrenar_modelo_con_parada_temprana(model_mlp, train_dataset, test_dataset)

Epoch 1/50


/Users/pedropablosanintrujillo/GitHub/proyecto-ml/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6463 - loss: 0.5790 - val_accuracy: 0.6825 - val_loss: 0.5210
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7947 - loss: 0.4047 - val_accuracy: 0.6864 - val_loss: 0.5576
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8768 - loss: 0.2675 - val_accuracy: 0.6898 - val_loss: 0.6286
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9272 - loss: 0.1714 - val_accuracy: 0.6986 - val_loss: 0.7606
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.
Mejor época según val_loss: 1 (val_loss = 0.5210, val_accuracy = 0.6825)


### 3.1 - Búsqueda de hiperparámetros 

## 3 - Red neuronal recurrente básica


Inicialmente, usaremos una red neuronal recurrente para clasificar las reseñas. Inicialmente, veremos el desempeño de una RNN simple, donde cada neurona tiene información sobre las operaciones que ha hecho en pasos anteriores. Las redes neuronales recurrentes funcionan bien al procesar lenguaje natural pues el lenguaje tiene una naturaleza secuencial.

Usaremos la implementación de RNNs dada por `tf.keras.layers.SimpleRNN`. Para que el texto pueda pasar exitosamente por una red neuronal, debemos vectorizarlo. Para esto, usaremos una capa inicial de la red neuronal dada por `tf.keras.layers.TextVectorization`. Este algoritmo de vectorización combina varias formas de vectorizar frases como _tf_idf_, _count_ (bag of words) y soporta ngramas.


In [76]:
# Para una red recurrente necesitamos una secuencia de índices de palabras (output_mode='int'),
# no un vector tf_idf. Cada reseña se rellena o recorta a 100 palabras (mediana: 57, máximo: 125).


# Tildes a quitar (unicodedata no sirve aquí porque Keras pasa tensores, no str,
# y TensorFlow no trae normalización Unicode sin el paquete tensorflow_text).
TILDES = {
    "[áàâäã]": "a",
    "[éèêë]": "e",
    "[íìîï]": "i",
    "[óòôöõ]": "o",
    "[úùûü]": "u",
    "ñ": "n",
    "ç": "c",
}


def standardize_text(text):
    text = tf.strings.lower(text, encoding="utf-8")
    text = tf.strings.regex_replace(text, "<br />", " ")
    for patron, letra in TILDES.items():
        text = tf.strings.regex_replace(text, patron, letra)
    text = tf.strings.regex_replace(text, "[^a-z']", " ")
    return text


def crear_encoder_rnn(train_df, n_gram=1):

    encoder_rnn = TextVectorization(
        output_mode="int",
        output_sequence_length=100,
        standardize=standardize_text,  ngrams=n_gram  # pyright: ignore[reportArgumentType]
    )
    encoder_rnn.adapt(train_df["text"])
    return encoder_rnn


encoder_rnn = crear_encoder_rnn(train_df)

texto_original = train_df["text"].iloc[0]
texto_vectorizado = encoder_rnn(tf.convert_to_tensor([texto_original]))

encoder_rnn = crear_encoder_rnn(train_df)

texto_original = train_df["text"][1]
texto_vectorizado = encoder_rnn(tf.convert_to_tensor([texto_original]))
print(f"Texto original: {texto_original}")
print(f"Texto vectorizado: {texto_vectorizado}")

Texto original: Les cuento como me fue. Compre esta tablet a mediados de año y llego en cuatro dias a mi casa en Guadalajara, sin ningun problema con el paquete. En mi opinion, la resolucion me resulto ser deficiente para ser honesto. La uso sobre todo por las tardes para revisar correos y ver videos mientras ceno. La pantalla tactil termino siendo muy buena en el uso real.
Texto vectorizado: [[  82  687   69   10  185   18   76 1860    6   70    3  104    9   13
     4   52   12    6   20   24    4  417   42  746  150   19    5   95
     4   20  435    2  530   10   71  107  998   14  107  437    2   23
   147   53   15   36  163   14  732 1686    9  496  938  356 7770    2
   270  570  195  219  172  481    4    5   23  378    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]]


Ahora, entrenaremos una red neuronal con una capa de embedding, una capa SimpleRNN de 64 neuronas y una capa oculta densa de 32 neuronas con función de activación ReLU. Asimismo, entrenamos un embedding con una dimensión de salida de 64. 


In [77]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(), dtype=tf.string),
        encoder_rnn,
        tf.keras.layers.Embedding(
            input_dim=encoder_rnn.vocabulary_size(), output_dim=64, mask_zero=True
        ),
        tf.keras.layers.SimpleRNN(64),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(3),
    ]
)

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_8            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_10 (Embedding)        │ (None, 100, 64)        │     1,127,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_14 (SimpleRNN)       │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,137,539 (4.34 MB)

 Trainable params: 1,137,539 (4.34 MB)

 Non-trainable params: 0 (0.00 B)

In [78]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

In [79]:
entrenar_modelo_con_parada_temprana(model, train_dataset, test_dataset)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.6530 - loss: 0.6225 - val_accuracy: 0.7587 - val_loss: 0.4584
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.8399 - loss: 0.3345 - val_accuracy: 0.7753 - val_loss: 0.4287
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9222 - loss: 0.1853 - val_accuracy: 0.7606 - val_loss: 0.5556
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9602 - loss: 0.0990 - val_accuracy: 0.7694 - val_loss: 0.6989
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9759 - loss: 0.0647 - val_accuracy: 0.7504 - val_loss: 0.7901
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.4287, val_accuracy = 0.7753)


### 2.1 - Probando diferentes combinaciones de capas

A continuación, probaremos 5 combinaciones distintas de capas para determinar cual tiene mejor desempeño en validación.

Para no repetir código, definimos una función que arma la red con la misma estructura de la celda anterior
(entrada de texto → `encoder_rnn` → `Embedding` → **capas recurrentes** → `Dense(32, relu)` → `Dense(3)`)
y solo cambia el bloque de capas recurrentes. Cada modelo tiene su propia capa de embedding, de modo que
ninguno comparte pesos entrenables con otro; `encoder_rnn` sí se comparte, pero no tiene pesos entrenables.

In [80]:
def construir_modelo_rnn(capas_recurrentes, nombre):
    """Arma y compila una red recurrente cuyo bloque central son las capas recibidas."""
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(), dtype=tf.string),
            encoder_rnn,
            tf.keras.layers.Embedding(
                input_dim=encoder_rnn.vocabulary_size(), output_dim=64, mask_zero=True
            ),
            *capas_recurrentes,
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(3),
        ],
        name=nombre,
    )
    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(),
        metrics=["accuracy"],
    )
    return model


# Guardamos el historial de entrenamiento de cada combinación para compararlas al final
historiales = {}

#### Modelo 1 - SimpleRNN bidireccional

La misma `SimpleRNN(64)` del modelo base, pero envuelta en `Bidirectional`: una copia lee la reseña de
izquierda a derecha y otra de derecha a izquierda, y sus salidas se concatenan. Así, la representación final
no depende solo de las últimas palabras de la reseña.

In [81]:
model_rnn_1 = construir_modelo_rnn(
    [tf.keras.layers.Bidirectional(tf.keras.layers.SimpleRNN(64))],
    nombre="simplernn_bidireccional",
)
model_rnn_1.summary()

historiales["1 - SimpleRNN bidireccional"] = entrenar_modelo_con_parada_temprana(
    model_rnn_1, train_dataset, test_dataset
)

Model: "simplernn_bidireccional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_8            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_11 (Embedding)        │ (None, 100, 64)        │     1,127,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_10                │ (None, 128)            │        16,512 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,147,843 (4.38 MB)

 Trainable params: 1,147,843 (4.38 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.6514 - loss: 0.6140 - val_accuracy: 0.7245 - val_loss: 0.4887
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - accuracy: 0.8477 - loss: 0.3245 - val_accuracy: 0.7650 - val_loss: 0.4484
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - accuracy: 0.9306 - loss: 0.1707 - val_accuracy: 0.7494 - val_loss: 0.6132
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9627 - loss: 0.0985 - val_accuracy: 0.7294 - val_loss: 0.8444
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9739 - loss: 0.0714 - val_accuracy: 0.7484 - val_loss: 0.7356
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.4484, val_accuracy = 0.7650)


#### Modelo 2 - Dos SimpleRNN apiladas

Dos capas `SimpleRNN` apiladas (64 y 32 neuronas). La primera usa `return_sequences=True` para entregarle a la
segunda la salida de cada paso de tiempo, no solo la del último.

In [82]:
model_rnn_2 = construir_modelo_rnn(
    [
        tf.keras.layers.SimpleRNN(64, return_sequences=True),
        tf.keras.layers.SimpleRNN(32),
    ],
    nombre="simplernn_apiladas",
)
model_rnn_2.summary()

historiales["2 - SimpleRNN apiladas"] = entrenar_modelo_con_parada_temprana(
    model_rnn_2, train_dataset, test_dataset
)

Model: "simplernn_apiladas"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_8            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_12 (Embedding)        │ (None, 100, 64)        │     1,127,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_16 (SimpleRNN)       │ (None, 100, 64)        │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_17 (SimpleRNN)       │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,139,619 (4.35 MB)

 Trainable params: 1,139,619 (4.35 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.5808 - loss: 0.7447 - val_accuracy: 0.7479 - val_loss: 0.4605
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - accuracy: 0.8198 - loss: 0.3584 - val_accuracy: 0.7728 - val_loss: 0.4351
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.9108 - loss: 0.2063 - val_accuracy: 0.7548 - val_loss: 0.5215
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9531 - loss: 0.1178 - val_accuracy: 0.7611 - val_loss: 0.6266
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9736 - loss: 0.0717 - val_accuracy: 0.7479 - val_loss: 0.7659
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.4351, val_accuracy = 0.7728)


#### Modelo 3 - LSTM

Reemplazamos la `SimpleRNN` por una `LSTM(64)`. Sus compuertas de entrada, olvido y salida le permiten conservar
información a lo largo de secuencias largas y mitigan el desvanecimiento del gradiente de la RNN simple.

In [83]:
model_rnn_3 = construir_modelo_rnn(
    [tf.keras.layers.LSTM(64)],
    nombre="lstm",
)
model_rnn_3.summary()

historiales["3 - LSTM"] = entrenar_modelo_con_parada_temprana(
    model_rnn_3, train_dataset, test_dataset
)

Model: "lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_8            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_13 (Embedding)        │ (None, 100, 64)        │     1,127,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,162,307 (4.43 MB)

 Trainable params: 1,162,307 (4.43 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.6378 - loss: 0.5891 - val_accuracy: 0.7665 - val_loss: 0.4457
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 15s 25ms/step - accuracy: 0.8279 - loss: 0.3270 - val_accuracy: 0.8163 - val_loss: 0.3236
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.8744 - loss: 0.2425 - val_accuracy: 0.8129 - val_loss: 0.3685
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.8973 - loss: 0.1957 - val_accuracy: 0.8188 - val_loss: 0.3653
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.9211 - loss: 0.1509 - val_accuracy: 0.8051 - val_loss: 0.4718
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.3236, val_accuracy = 0.8163)


#### Modelo 4 - GRU bidireccional con dropout

Una `GRU(64)` bidireccional (similar a la LSTM pero con menos parámetros) seguida de un `Dropout(0.5)` para
regularizar, ya que con alrededor de 18.000 reseñas de entrenamiento los modelos se sobreajustan con facilidad.

In [84]:
model_rnn_4 = construir_modelo_rnn(
    [
        tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64)),
        tf.keras.layers.Dropout(0.5),
    ],
    nombre="gru_bidireccional_dropout",
)
model_rnn_4.summary()

historiales["4 - GRU bidireccional + dropout"] = entrenar_modelo_con_parada_temprana(
    model_rnn_4, train_dataset, test_dataset
)

Model: "gru_bidireccional_dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_8            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_14 (Embedding)        │ (None, 100, 64)        │     1,127,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_11                │ (None, 128)            │        49,920 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,181,251 (4.51 MB)

 Trainable params: 1,181,251 (4.51 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 28s 46ms/step - accuracy: 0.6181 - loss: 0.6362 - val_accuracy: 0.7494 - val_loss: 0.4689
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 27s 47ms/step - accuracy: 0.8356 - loss: 0.3248 - val_accuracy: 0.8124 - val_loss: 0.3325
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 28s 49ms/step - accuracy: 0.8909 - loss: 0.2258 - val_accuracy: 0.8158 - val_loss: 0.3944
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 27s 48ms/step - accuracy: 0.9219 - loss: 0.1705 - val_accuracy: 0.8192 - val_loss: 0.4142
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 29s 50ms/step - accuracy: 0.9449 - loss: 0.1247 - val_accuracy: 0.8192 - val_loss: 0.5305
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.3325, val_accuracy = 0.8124)


#### Modelo 5 - Dos LSTM bidireccionales apiladas con dropout

La combinación más profunda: dos `LSTM` bidireccionales apiladas (64 y 32 neuronas), con `Dropout(0.3)` entre ellas
y `Dropout(0.5)` antes de las capas densas.

In [85]:
model_rnn_5 = construir_modelo_rnn(
    [
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
        tf.keras.layers.Dropout(0.5),
    ],
    nombre="lstm_bidireccionales_apiladas",
)
model_rnn_5.summary()

historiales["5 - LSTM bidireccionales apiladas"] = entrenar_modelo_con_parada_temprana(
    model_rnn_5, train_dataset, test_dataset
)

Model: "lstm_bidireccionales_apiladas"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_8            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_15 (Embedding)        │ (None, 100, 64)        │     1,127,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_12                │ (None, 100, 128)       │        66,048 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_13                │ (None, 64)             │        41,216 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,236,547 (4.72 MB)

 Trainable params: 1,236,547 (4.72 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 52s 85ms/step - accuracy: 0.6093 - loss: 0.6118 - val_accuracy: 0.6575 - val_loss: 0.5294
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 51s 88ms/step - accuracy: 0.6664 - loss: 0.5065 - val_accuracy: 0.7621 - val_loss: 0.4604
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 51s 89ms/step - accuracy: 0.8133 - loss: 0.3513 - val_accuracy: 0.7978 - val_loss: 0.3681
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 50s 87ms/step - accuracy: 0.8689 - loss: 0.2610 - val_accuracy: 0.8075 - val_loss: 0.3969
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 49s 85ms/step - accuracy: 0.8866 - loss: 0.2158 - val_accuracy: 0.7973 - val_loss: 0.4596
Epoch 6/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 50s 87ms/step - accuracy: 0.9091 - loss: 0.1706 - val_accuracy: 0.8002 - val_loss: 0.5447
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.
Mejor época según val_loss: 3 (val_loss = 0.3681, val_accuracy = 0.7978)


#### Creación de archivos submission



In [86]:
# Métricas de la época que restauró la parada temprana (la de menor val_loss)
resumen_rnn = pd.DataFrame(
    [
        {
            "modelo": nombre,
            "mejor_epoca": h.mejor_epoca + 1,
            "val_loss": h.history["val_loss"][h.mejor_epoca],
            "val_accuracy": h.history["val_accuracy"][h.mejor_epoca],
            "train_accuracy": h.history["accuracy"][h.mejor_epoca],
        }
        for nombre, h in historiales.items()
    ]
).sort_values("val_accuracy", ascending=False)

resumen_rnn

,modelo,mejor_epoca,val_loss,val_accuracy,train_accuracy
2,3 - LSTM,2,0.323592,0.816317,0.827949
3,4 - GRU bidireccional + dropout,2,0.332540,0.812408,0.835605
4,5 - LSTM bidireccionales apiladas,3,0.368050,0.797753,0.813345
1,2 - SimpleRNN apiladas,2,0.435076,0.772838,0.819806
0,1 - SimpleRNN bidireccional,2,0.448385,0.765022,0.847657


In [87]:
def crear_archivo_predicciones(
    model, df, archivo_salida="./datos/submission.csv", columnas=("text",)
):
    """Predice la etiqueta de cada reseña de `df` y guarda un CSV con el formato de
    ./datos/sample_submission.csv: columnas `id` y `answer` ('negativo', 'neutral' o 'positivo').

    `columnas` son las columnas de `df` que recibe el modelo: con una sola (por defecto "text")
    se le pasan los textos directamente; con varias, un dict {nombre_columna: textos}, que es lo
    que esperan los modelos de dos entradas de la sección 3.2."""
    if len(columnas) == 1:
        entradas = df[columnas[0]].values
    else:
        entradas = {c: df[c].values for c in columnas}
    textos = tf.data.Dataset.from_tensor_slices(entradas).batch(32)

    # El modelo devuelve logits (una fila de 3 valores por reseña): la clase predicha es la de
    # mayor logit, y la traducimos de vuelta a texto con la misma lista usada al codificarlas.
    logits = model.predict(textos)
    clases = logits.argmax(axis=1)

    submission = pd.DataFrame(
        {"id": df["id"].values, "answer": [etiquetas[c] for c in clases]}
    )
    submission.to_csv(archivo_salida, index=False)

    print(f"Archivo guardado en {archivo_salida} ({len(submission)} predicciones)")
    print(submission["answer"].value_counts(normalize=True))
    return submission

In [88]:
# Cambiar por el modelo con mejor val_accuracy
modelos_rnn = [model_rnn_1, model_rnn_2, model_rnn_3, model_rnn_4, model_rnn_5]
nombres_modelo = ["Simple RNN", "Simple RNNs apiladas","LSTM","GRU","LSTM apiladas"]
modelos_rnn = {"nombre": nombres_modelo, "modelo": modelos_rnn}

for nombre, modelo in zip(modelos_rnn["nombre"], modelos_rnn["modelo"]):
    submission = crear_archivo_predicciones(modelo, dataset_eval, f"./datos/submission_{nombre}.csv")
    submission.head()

212/212 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Archivo guardado en ./datos/submission_Simple RNN.csv (6781 predicciones)
answer
positivo    0.536204
negativo    0.269429
neutral     0.194367
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Archivo guardado en ./datos/submission_Simple RNNs apiladas.csv (6781 predicciones)
answer
negativo    0.450229
positivo    0.351423
neutral     0.198348
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
Archivo guardado en ./datos/submission_LSTM.csv (6781 predicciones)
answer
positivo    0.454063
negativo    0.356290
neutral     0.189648
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step
Archivo guardado en ./datos/submission_GRU.csv (6781 predicciones)
answer
negativo    0.404070
positivo    0.403333
neutral     0.192597
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step
Archivo guardado en ./datos/submission_LSTM apiladas.csv (6781 predicciones)
answer
nega

### 3.2 - Ajustando y regularizando los modelos RNN

En el proyecto anterior, la última oración era la que más significado daba a la reseña completa. Para esto, agregabamos la última oración como una entrada adicional al modelo. En este caso, haremos lo mismo.

Asimismo, vemos que los modelos que estamos usando tienden al overfitting. Para solucionar esto intentaremos pararemos el entrenamiento en las epochs en función de la función de pérdida y no el accuracy en el entrenamiento. De esta forma, el modelo elegido no se elige de acuerdo a los datos específicos de entrenamiento. 

También, intentaremos regularizar más las RNNs. Para esto, usaremos ``SpatialDropout1D`` después del embedding. A diferencia de ``Dropout``, ``SpatialDropout1D`` hace dropout para columnas enteras del espacio, lo que aumenta la regularización.



In [89]:
import re

def ultima_oracion(texto):
    """Devuelve la última oración de la reseña (corta después de . ! ?)."""
    oraciones = [o for o in re.split(r"(?<=[.!?])\s+", texto.strip()) if o]
    return oraciones[-1] if oraciones else texto

# .assign crea un DataFrame nuevo y evita el SettingWithCopyWarning
train_df = train_df.assign(ultima=train_df["text"].map(ultima_oracion))
test_df = test_df.assign(ultima=test_df["text"].map(ultima_oracion))
dataset_eval = dataset_eval.assign(ultima=dataset_eval["text"].map(ultima_oracion))


def entradas_modelo(df, columnas):
    """Una columna -> arreglo de textos (como antes); varias -> dict {nombre_columna: textos}."""
    if len(columnas) == 1:
        return df[columnas[0]].values
    return {c: df[c].values for c in columnas}


def dataset_a_tf(df, shuffle, columnas=("text",)):
    ds = tf.data.Dataset.from_tensor_slices(
        (entradas_modelo(df, columnas), df["label"].map(etiqueta_a_id).values)
    )
    if shuffle:
        ds = ds.shuffle(len(df), seed=42)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)


train_dataset = dataset_a_tf(train_df, shuffle=True)
test_dataset = dataset_a_tf(test_df, shuffle=False)



Para comparar de forma justa, cada modelo mejorado conserva exactamente el bloque recurrente de su
modelo original de la sección 2.1. Solo cambian dos cosas:

1. Una segunda entrada con la última oración de la reseña, procesada por su propia copia del bloque recurrente.
2. `SpatialDropout1D(0.3)` después del embedding.

La parada temprana según `val_loss` ya no es una diferencia entre ambos grupos: `entrenar_modelo_con_parada_temprana`
la usa por defecto, así que los modelos originales también se eligen con la pérdida de validación.

In [90]:
def construir_modelo_rnn_mejorado(
    crear_capas_recurrentes, nombre, tasa_spatial_dropout=0.3
):
    """Igual que `construir_modelo_rnn`, pero con las mejoras de esta sección:

    - Dos entradas: la reseña completa (`text`) y su última oración (`ultima`). Cada una pasa
      por su propia copia del bloque recurrente y las dos salidas se concatenan antes de las
      capas densas.
    - `SpatialDropout1D` justo después del embedding, en ambas ramas.

    `crear_capas_recurrentes` es una función que devuelve una lista NUEVA de capas cada vez que se
    llama, para que cada rama tenga sus propios pesos (si se usaran las mismas capas en las dos
    ramas, Keras compartiría sus pesos). El embedding sí se comparte entre ambas ramas, de modo
    que las palabras de la última oración también aprenden de la reseña completa.
    """
    embedding = tf.keras.layers.Embedding(
        input_dim=encoder_rnn.vocabulary_size(), output_dim=64, mask_zero=True
    )

    def rama(entrada):
        x = embedding(encoder_rnn(entrada))
        x = tf.keras.layers.SpatialDropout1D(tasa_spatial_dropout)(x)
        for capa in crear_capas_recurrentes():
            x = capa(x)
        return x

    # Los nombres de los Input coinciden con las columnas del DataFrame (las claves del dict)
    texto = tf.keras.layers.Input(shape=(), dtype=tf.string, name="text")
    ultima = tf.keras.layers.Input(shape=(), dtype=tf.string, name="ultima")

    x = tf.keras.layers.Concatenate()([rama(texto), rama(ultima)])
    x = tf.keras.layers.Dense(32, activation="relu")(x)
    salida = tf.keras.layers.Dense(3)(x)

    model = tf.keras.Model([texto, ultima], salida, name=nombre)
    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(),
        metrics=["accuracy"],
    )
    return model


# Conjuntos de datos con las dos entradas: reseña completa y última oración
columnas_2 = ("text", "ultima")
train_dataset_2 = dataset_a_tf(train_df, shuffle=True, columnas=columnas_2)
test_dataset_2 = dataset_a_tf(test_df, shuffle=False, columnas=columnas_2)

# Usamos las mismas claves que en `historiales` para poder comparar cada modelo con su original
historiales_mejorados = {}

#### Modelo 1 mejorado - SimpleRNN bidireccional

Mismo bloque recurrente que el modelo 1: una `SimpleRNN(64)` bidireccional.

In [91]:
model_rnn_1_mejorado = construir_modelo_rnn_mejorado(
    lambda: [tf.keras.layers.Bidirectional(tf.keras.layers.SimpleRNN(64))],
    nombre="simplernn_bidireccional_mejorado",
)
model_rnn_1_mejorado.summary()

historiales_mejorados["1 - SimpleRNN bidireccional"] = entrenar_modelo_con_parada_temprana(
    model_rnn_1_mejorado, train_dataset_2, test_dataset_2, monitor="val_loss"
)

Model: "simplernn_bidireccional_mejorado"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ultima (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 100)       │          0 │ text[0][0],       │
│ (TextVectorization) │                   │            │ ultima[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_16        │ (None, 100, 64)   │  1,127,104 │ text_vectorizati… │
│ (Embedding)         │                   │            │ text_vectorizati… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_8 │ (None, 100, 64)   │          0 │ embedding_16[0][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_20        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_9 │ (None, 100, 64)   │          0 │ embedding_16[1][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_21        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_14    │ (None, 128)       │     16,512 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal_20[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_15    │ (None, 128)       │     16,512 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal_21[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 256)       │          0 │ bidirectional_14… │
│ (Concatenate)       │                   │            │ bidirectional_15… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_38 (Dense)    │ (None, 32)        │      8,224 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_39 (Dense)    │ (None, 3)         │         99 │ dense_38[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,168,451 (4.46 MB)

 Trainable params: 1,168,451 (4.46 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50


/Users/pedropablosanintrujillo/GitHub/proyecto-ml/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


576/576 ━━━━━━━━━━━━━━━━━━━━ 24s 36ms/step - accuracy: 0.6313 - loss: 0.6678 - val_accuracy: 0.7694 - val_loss: 0.4374
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.8182 - loss: 0.3551 - val_accuracy: 0.7640 - val_loss: 0.4282
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 20s 34ms/step - accuracy: 0.8939 - loss: 0.2414 - val_accuracy: 0.7743 - val_loss: 0.4434
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.9246 - loss: 0.1698 - val_accuracy: 0.7655 - val_loss: 0.5536
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - accuracy: 0.9484 - loss: 0.1240 - val_accuracy: 0.7562 - val_loss: 0.7375
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.4282, val_accuracy = 0.7640)


#### Modelo 2 mejorado - Dos SimpleRNN apiladas

Mismo bloque recurrente que el modelo 2: `SimpleRNN(64, return_sequences=True)` seguida de `SimpleRNN(32)`.

In [92]:
model_rnn_2_mejorado = construir_modelo_rnn_mejorado(
    lambda: [
        tf.keras.layers.SimpleRNN(64, return_sequences=True),
        tf.keras.layers.SimpleRNN(32),
    ],
    nombre="simplernn_apiladas_mejorado",
)
model_rnn_2_mejorado.summary()

historiales_mejorados["2 - SimpleRNN apiladas"] = entrenar_modelo_con_parada_temprana(
    model_rnn_2_mejorado, train_dataset_2, test_dataset_2, monitor="val_loss"
)

Model: "simplernn_apiladas_mejorado"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ultima (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 100)       │          0 │ text[0][0],       │
│ (TextVectorization) │                   │            │ ultima[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_17        │ (None, 100, 64)   │  1,127,104 │ text_vectorizati… │
│ (Embedding)         │                   │            │ text_vectorizati… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_… │ (None, 100, 64)   │          0 │ embedding_17[0][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_22        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_… │ (None, 100, 64)   │          0 │ embedding_17[1][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_23        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_20       │ (None, 100, 64)   │      8,256 │ spatial_dropout1… │
│ (SimpleRNN)         │                   │            │ not_equal_22[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_22       │ (None, 100, 64)   │      8,256 │ spatial_dropout1… │
│ (SimpleRNN)         │                   │            │ not_equal_23[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_21       │ (None, 32)        │      3,104 │ simple_rnn_20[0]… │
│ (SimpleRNN)         │                   │            │ not_equal_22[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_23       │ (None, 32)        │      3,104 │ simple_rnn_22[0]… │
│ (SimpleRNN)         │                   │            │ not_equal_23[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 64)        │          0 │ simple_rnn_21[0]… │
│ (Concatenate)       │                   │            │ simple_rnn_23[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_40 (Dense)    │ (None, 32)        │      2,080 │ concatenate_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_41 (Dense)    │ (None, 3)         │         99 │ dense_40[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,152,003 (4.39 MB)

 Trainable params: 1,152,003 (4.39 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - accuracy: 0.6079 - loss: 0.6940 - val_accuracy: 0.7738 - val_loss: 0.4374
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 19s 33ms/step - accuracy: 0.8189 - loss: 0.3673 - val_accuracy: 0.7841 - val_loss: 0.4160
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 18s 32ms/step - accuracy: 0.8855 - loss: 0.2580 - val_accuracy: 0.7767 - val_loss: 0.4725
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 19s 33ms/step - accuracy: 0.9268 - loss: 0.1787 - val_accuracy: 0.7518 - val_loss: 0.5850
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 19s 33ms/step - accuracy: 0.9506 - loss: 0.1258 - val_accuracy: 0.7704 - val_loss: 0.6381
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.4160, val_accuracy = 0.7841)


#### Modelo 3 mejorado - LSTM

Mismo bloque recurrente que el modelo 3: una `LSTM(64)`.

In [93]:
model_rnn_3_mejorado = construir_modelo_rnn_mejorado(
    lambda: [tf.keras.layers.LSTM(64)],
    nombre="lstm_mejorado",
)
model_rnn_3_mejorado.summary()

historiales_mejorados["3 - LSTM"] = entrenar_modelo_con_parada_temprana(
    model_rnn_3_mejorado, train_dataset_2, test_dataset_2, monitor="val_loss"
)

Model: "lstm_mejorado"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ultima (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 100)       │          0 │ text[0][0],       │
│ (TextVectorization) │                   │            │ ultima[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_18        │ (None, 100, 64)   │  1,127,104 │ text_vectorizati… │
│ (Embedding)         │                   │            │ text_vectorizati… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_… │ (None, 100, 64)   │          0 │ embedding_18[0][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_24        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_… │ (None, 100, 64)   │          0 │ embedding_18[1][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_25        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ (None, 64)        │     33,024 │ spatial_dropout1… │
│                     │                   │            │ not_equal_24[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ (None, 64)        │     33,024 │ spatial_dropout1… │
│                     │                   │            │ not_equal_25[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 128)       │          0 │ lstm_6[0][0],     │
│ (Concatenate)       │                   │            │ lstm_7[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_42 (Dense)    │ (None, 32)        │      4,128 │ concatenate_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_43 (Dense)    │ (None, 3)         │         99 │ dense_42[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,197,379 (4.57 MB)

 Trainable params: 1,197,379 (4.57 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 24s 39ms/step - accuracy: 0.6862 - loss: 0.5395 - val_accuracy: 0.7924 - val_loss: 0.3815
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 23s 40ms/step - accuracy: 0.8397 - loss: 0.3035 - val_accuracy: 0.8007 - val_loss: 0.3450
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 22s 37ms/step - accuracy: 0.8689 - loss: 0.2520 - val_accuracy: 0.7914 - val_loss: 0.3626
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 21s 37ms/step - accuracy: 0.8903 - loss: 0.2162 - val_accuracy: 0.8021 - val_loss: 0.4503
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 22s 38ms/step - accuracy: 0.9043 - loss: 0.1795 - val_accuracy: 0.7997 - val_loss: 0.4839
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.3450, val_accuracy = 0.8007)


#### Modelo 4 mejorado - GRU bidireccional con dropout

Mismo bloque recurrente que el modelo 4: una `GRU(64)` bidireccional seguida de `Dropout(0.5)`.

In [94]:
model_rnn_4_mejorado = construir_modelo_rnn_mejorado(
    lambda: [
        tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64)),
        tf.keras.layers.Dropout(0.5),
    ],
    nombre="gru_bidireccional_dropout_mejorado",
)
model_rnn_4_mejorado.summary()

historiales_mejorados["4 - GRU bidireccional + dropout"] = entrenar_modelo_con_parada_temprana(
    model_rnn_4_mejorado, train_dataset_2, test_dataset_2, monitor="val_loss"
)

Model: "gru_bidireccional_dropout_mejorado"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ultima (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 100)       │          0 │ text[0][0],       │
│ (TextVectorization) │                   │            │ ultima[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_19        │ (None, 100, 64)   │  1,127,104 │ text_vectorizati… │
│ (Embedding)         │                   │            │ text_vectorizati… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_… │ (None, 100, 64)   │          0 │ embedding_19[0][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_26        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_… │ (None, 100, 64)   │          0 │ embedding_19[1][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_27        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_16    │ (None, 128)       │     49,920 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal_26[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_17    │ (None, 128)       │     49,920 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal_27[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 128)       │          0 │ bidirectional_16… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 128)       │          0 │ bidirectional_17… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_7       │ (None, 256)       │          0 │ dropout_6[0][0],  │
│ (Concatenate)       │                   │            │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_44 (Dense)    │ (None, 32)        │      8,224 │ concatenate_7[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_45 (Dense)    │ (None, 3)         │         99 │ dense_44[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,235,267 (4.71 MB)

 Trainable params: 1,235,267 (4.71 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 41s 67ms/step - accuracy: 0.6928 - loss: 0.5626 - val_accuracy: 0.8012 - val_loss: 0.3602
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 42s 73ms/step - accuracy: 0.8451 - loss: 0.2985 - val_accuracy: 0.7978 - val_loss: 0.3487
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 43s 75ms/step - accuracy: 0.8744 - loss: 0.2445 - val_accuracy: 0.8041 - val_loss: 0.3802
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 42s 73ms/step - accuracy: 0.8980 - loss: 0.2015 - val_accuracy: 0.8041 - val_loss: 0.4163
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 43s 74ms/step - accuracy: 0.9207 - loss: 0.1643 - val_accuracy: 0.8129 - val_loss: 0.4005
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.3487, val_accuracy = 0.7978)


#### Modelo 5 mejorado - Dos LSTM bidireccionales apiladas con dropout

Mismo bloque recurrente que el modelo 5: dos `LSTM` bidireccionales apiladas (64 y 32 neuronas) con `Dropout(0.3)` entre ellas y `Dropout(0.5)` al final.

In [95]:
model_rnn_5_mejorado = construir_modelo_rnn_mejorado(
    lambda: [
        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64, return_sequences=True)
        ),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
        tf.keras.layers.Dropout(0.5),
    ],
    nombre="lstm_bidireccionales_apiladas_mejorado",
)
model_rnn_5_mejorado.summary()

historiales_mejorados["5 - LSTM bidireccionales apiladas"] = entrenar_modelo_con_parada_temprana(
    model_rnn_5_mejorado, train_dataset_2, test_dataset_2, monitor="val_loss"
)

Model: "lstm_bidireccionales_apiladas_mejorado"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ultima (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 100)       │          0 │ text[0][0],       │
│ (TextVectorization) │                   │            │ ultima[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_20        │ (None, 100, 64)   │  1,127,104 │ text_vectorizati… │
│ (Embedding)         │                   │            │ text_vectorizati… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_… │ (None, 100, 64)   │          0 │ embedding_20[0][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_28        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_… │ (None, 100, 64)   │          0 │ embedding_20[1][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_29        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_18    │ (None, 100, 128)  │     66,048 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal_28[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_20    │ (None, 100, 128)  │     66,048 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal_29[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 100, 128)  │          0 │ bidirectional_18… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 100, 128)  │          0 │ bidirectional_20… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_19    │ (None, 64)        │     41,216 │ dropout_8[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal_28[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_21    │ (None, 64)        │     41,216 │ dropout_10[0][0], │
│ (Bidirectional)     │                   │            │ not_equal_29[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 64)        │          0 │ bidirectional_19… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 64)        │          0 │ bidirectional_21… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_8       │ (None, 128)       │          0 │ dropout_9[0][0],  │
│ (Concatenate)       │                   │            │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_46 (Dense)    │ (None, 32)        │      4,128 │ concatenate_8[0]… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,345,859 (5.13 MB)

 Trainable params: 1,345,859 (5.13 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 78s 125ms/step - accuracy: 0.6852 - loss: 0.5395 - val_accuracy: 0.7811 - val_loss: 0.3572
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 77s 133ms/step - accuracy: 0.8340 - loss: 0.3177 - val_accuracy: 0.7948 - val_loss: 0.3515
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 78s 136ms/step - accuracy: 0.8636 - loss: 0.2594 - val_accuracy: 0.7963 - val_loss: 0.3874
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 78s 135ms/step - accuracy: 0.8830 - loss: 0.2253 - val_accuracy: 0.7934 - val_loss: 0.4246
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 78s 135ms/step - accuracy: 0.8919 - loss: 0.2066 - val_accuracy: 0.7909 - val_loss: 0.4880
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.3515, val_accuracy = 0.7948)


#### Comparación con los modelos originales

Para cada modelo tomamos las métricas de la época que restauró la parada temprana. Tanto en los originales como
en los mejorados esa época se eligió con `val_loss`, así que el `val_accuracy` reportado no está sesgado por la
selección y ambos grupos se comparan en igualdad de condiciones. Con ~2.000 reseñas de validación, el error estándar
del accuracy es de casi 1 punto, así que diferencias menores a eso no son concluyentes.

In [96]:
def resumir_historiales(historiales):
    """Una fila por modelo con las métricas de la época restaurada por la parada temprana."""
    filas = []
    for nombre, h in historiales.items():
        mejor = h.mejor_epoca
        filas.append(
            {
                "modelo": nombre,
                "mejor_epoca": mejor + 1,
                "val_accuracy": h.history["val_accuracy"][mejor],
                "val_loss": h.history["val_loss"][mejor],
                "train_accuracy": h.history["accuracy"][mejor],
            }
        )
    return pd.DataFrame(filas)


comparacion = resumir_historiales(historiales).merge(
    resumir_historiales(historiales_mejorados),
    on="modelo",
    suffixes=("_original", "_mejorado"),
)
comparacion["diferencia_val_accuracy"] = (
    comparacion["val_accuracy_mejorado"] - comparacion["val_accuracy_original"]
)

comparacion[
    [
        "modelo",
        "mejor_epoca_original",
        "mejor_epoca_mejorado",
        "val_loss_original",
        "val_loss_mejorado",
        "val_accuracy_original",
        "val_accuracy_mejorado",
        "diferencia_val_accuracy",
        "train_accuracy_original",
        "train_accuracy_mejorado",
    ]
].sort_values("val_accuracy_mejorado", ascending=False)

,modelo,mejor_epoca_original,mejor_epoca_mejorado,val_loss_original,val_loss_mejorado,val_accuracy_original,val_accuracy_mejorado,diferencia_val_accuracy,train_accuracy_original,train_accuracy_mejorado
2,3 - LSTM,2,2,0.323592,0.345020,0.816317,0.800684,-0.015633,0.827949,0.839676
3,4 - GRU bidireccional + dropout,2,2,0.332540,0.348663,0.812408,0.797753,-0.014656,0.835605,0.845106
4,5 - LSTM bidireccionales apiladas,3,2,0.368050,0.351484,0.797753,0.794822,-0.002931,0.813345,0.834030
1,2 - SimpleRNN apiladas,2,2,0.435076,0.415954,0.772838,0.784074,0.011236,0.819806,0.818883
0,1 - SimpleRNN bidireccional,2,2,0.448385,0.428196,0.765022,0.764045,-0.000977,0.847657,0.818231


In [97]:
# Archivos submission de los modelos mejorados (reciben la reseña y su última oración)
modelos_mejorados = {
    "Simple RNN": model_rnn_1_mejorado,
    "Simple RNNs apiladas": model_rnn_2_mejorado,
    "LSTM": model_rnn_3_mejorado,
    "GRU": model_rnn_4_mejorado,
    "LSTM apiladas": model_rnn_5_mejorado,
}

for nombre, modelo in modelos_mejorados.items():
    crear_archivo_predicciones(
        modelo,
        dataset_eval,
        f"./datos/submission_{nombre}_mejorado.csv",
        columnas=columnas_2,
    )

212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
Archivo guardado en ./datos/submission_Simple RNN_mejorado.csv (6781 predicciones)
answer
positivo    0.494470
negativo    0.310869
neutral     0.194662
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step
Archivo guardado en ./datos/submission_Simple RNNs apiladas_mejorado.csv (6781 predicciones)
answer
negativo    0.484589
positivo    0.314703
neutral     0.200708
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step
Archivo guardado en ./datos/submission_LSTM_mejorado.csv (6781 predicciones)
answer
negativo    0.482230
positivo    0.328123
neutral     0.189648
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step
Archivo guardado en ./datos/submission_GRU_mejorado.csv (6781 predicciones)
answer
negativo    0.452441
positivo    0.355552
neutral     0.192007
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step
Archivo guardado en ./datos/submission_LSTM apilada

Vemos entonces que estas estrategias de regularización no funcionaron.

## 4 - Transformers